### Motivation
Suppose three models are trained:
| Model               | Validation Accuracy |
| ------------------- | ------------------- |
| Logistic Regression | 88%                 |
| Random Forest       | 91%                 |
| XGBoost             | 93%                 |

Instead of selecting one model:
```
Choose the strengths of all three.
```
This is the foundation of stacking.

### Ensemble Categories Recap
#### Bagging
```
Same model type
Independent training
```
Example:
```
Random Forest
```

#### Boosting
```
Same model type
Sequential training
```
Example:
```
XGBoost
```

#### Stacking
```
Different model types
Meta-learning
```
Example:
```
Logistic Regression
Random Forest
XGBoost
↓
Meta Model
```

### What is Stacking?
Stacking means:
> Train multiple base models and train another model to combine their predictions.

The combining model is called: **Meta Model**

(or Level-2 Model)

### High-Level Architecture
```
Training Data
    ↓
 ┌─────────────┐
 │ LogisticReg │
 ├─────────────┤
 │ RandomForest│
 ├─────────────┤
 │ XGBoost     │
 └─────────────┘
    ↓
 Predictions
    ↓
 Meta Model
    ↓
 Final Prediction
```

### Why Stacking Works
Different models make different mistakes.

Example:

#### Logistic Regression
Captures:
```
Linear patterns
```
Misses:
```
Complex interactions
```

#### Random Forest
Captures:
```
Non-linear interactions
```
Misses:
```
Some global trends
```

#### XGBoost
Captures:
```
Residual patterns
```
Meta model learns:
```
Which model to trust
```

### Naive Stacking Problem
Suppose:

Train base models.

Then use same training predictions to train meta-model.

This causes: **Severe Leakage**

Because meta-model sees predictions from models trained on same data.

### Correct Stacking Procedure

Use: **Out-of-Fold (OOF) Predictions**

#### Example
5-fold CV.

#### Fold 1
Train:
```
Folds 2–5
```
Predict:
```
Fold 1
```

#### Fold 2
Train:
```
Folds 1,3,4,5
```
Predict:
```
Fold 2
```
Continue.

Now every training sample has:
```
Prediction from model
that never saw it
```
This prevents leakage.

### Level-0 and Level-1 Models
#### Level-0
Base learners.

Example:
```
Logistic Regression
Random Forest
XGBoost
```

#### Level-1
Meta learner.

Example:
```
Logistic Regression
```
often used.

### Example
Suppose:

Three models output probabilities.

#### Sample A
| Model    | Prediction |
| -------- | ---------- |
| Logistic | 0.70       |
| RF       | 0.60       |
| XGB      | 0.95       |

Meta model input:
```
[0.70, 0.60, 0.95]
```
Output:
```
Final prediction
```

### Mathematical View
Base models:
$$
f_1(x), f_2(x), f_3(x)
$$

Meta model:
$$
g(f_1(x), f_2(x), f_3(x))
$$

Final output:
$$
\hat y = g(...)
$$

### sklearn Stacking Example
#### Classification
```python
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

estimators = [
    ("rf", RandomForestClassifier()),
    ("xgb", XGBClassifier())
]

stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression()
)

stack.fit(X_train, y_train)
```

#### Regression
```python
from sklearn.ensemble import StackingRegressor
```

### What is Blending?
Blending is a simpler version of stacking.

#### Stacking
Uses:
```
K-Fold OOF Predictions
```

#### Blending
Uses:
```
Holdout Validation Set
```
Example:
```
Train Set
Validation Set
```
Train base models.

Generate predictions on validation set.

Use them to train meta-model.

### Blending Workflow
```
Train Base Models
    ↓
Predict Validation Set
    ↓
Meta Model Training
    ↓
Final Prediction
```

### Stacking vs Blending
| Feature               | Stacking       | Blending        |
| --------------------- | -------------- | --------------- |
| Uses CV               | Yes            | No              |
| Leakage Risk          | Lower          | Slightly Higher |
| Computational Cost    | Higher         | Lower           |
| Accuracy              | Usually Better | Slightly Lower  |
| Production Popularity | Higher         | Moderate        |

### Advantages
- Often highest accuracy
- Leverages multiple algorithms
- Reduces model-specific weaknesses
- Extremely powerful for tabular data

### Limitations
- Complex pipeline
- Longer training
- Harder debugging
- More difficult deployment
- Increased inference latency